# PriMock57 Exploratory Data Analysis
**Medical Voice Scribe BTP Project**

This notebook evaluates the **PriMock57** dataset of primary care mock doctor-patient consultations.

In [ ]:
import os
import json
from pathlib import Path
import soundfile as sf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
base_dir = Path('../../voice_scribe_data/primock57')
audio_dir = base_dir / 'audio'
print("Scanning PriMock57 audio directory:", audio_dir)
wavs = sorted(audio_dir.glob('*.wav'))
print(f"Total audio tracks found: {len(wavs)}")


## 1. Dual-Channel Doctor & Patient Consultation Pairing
Investigating duration synchronization between doctor and patient tracks.

In [ ]:
consultations = {}
for w in wavs:
    name = w.stem
    if name.endswith('_doctor'):
        cid = name[:-7]
        consultations.setdefault(cid, {})['doctor'] = w
    elif name.endswith('_patient'):
        cid = name[:-8]
        consultations.setdefault(cid, {})['patient'] = w

print(f"Total paired consultations: {len(consultations)}")

durations = []
for cid, tracks in consultations.items():
    if 'doctor' in tracks and 'patient' in tracks:
        d_dur = sf.info(tracks['doctor']).duration
        p_dur = sf.info(tracks['patient']).duration
        durations.append({'cid': cid, 'doc_dur': d_dur, 'pat_dur': p_dur, 'diff': abs(d_dur - p_dur)})

df_dur = pd.DataFrame(durations)
print(df_dur[['doc_dur', 'pat_dur', 'diff']].describe())


## 2. TextGrid Alignment & Utterance Distribution

In [ ]:
from collections import Counter
tg_dir = base_dir / 'transcripts'
tgs = sorted(tg_dir.glob('*.TextGrid'))
print(f"Total TextGrid alignment files: {len(tgs)}")

# Display sample lines from the first TextGrid
sample_tg = tgs[0].read_text(errors='ignore').splitlines()
print("\n".join(sample_tg[:20]))


## 3. Audio & Spectrogram Visualization

In [ ]:
fig_path = Path('../figures/primock57/primock57_sample_waveform_spectrogram.png')
if fig_path.exists():
    from IPython.display import Image, display
    display(Image(filename=str(fig_path)))
